# OOD Prompt Metrics to Drive (Self-contained)

This notebook runs LLM prompt-metric inference for OOD data in Colab and saves output CSV to Google Drive.

Default model is `mistralai/Mistral-7B-Instruct-v0.2`.


In [ ]:
%pip install -q torch transformers accelerate bitsandbytes pandas tqdm huggingface_hub


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

MY_DRIVE_SUBDIR = 'BeyondFK'  # change if needed
BASE_DIR = os.path.join('/content/drive/MyDrive', MY_DRIVE_SUBDIR)
OOD_DIR = os.path.join(BASE_DIR, 'ood')
PROMPT_DIR = os.path.join(BASE_DIR, 'outputs', 'prompt_metrics')
os.makedirs(OOD_DIR, exist_ok=True)

INPUT_CSV = os.path.join(OOD_DIR, 'onestopenglish_with_static.csv')
PROMPT_JSON = os.path.join(PROMPT_DIR, 'prompt_questions.json')
OUTPUT_CSV = os.path.join(OOD_DIR, 'onestopenglish_features_mistral-7b.csv')
PROGRESS_CSV = os.path.join(OOD_DIR, 'mistral-7b_ood_prompt_progress.csv')

MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.2'
MODEL_TAG = 'mistral-7b'
TEXT_COL = 'full_text'
BATCH_SIZE = 4
SAVE_EVERY = 10
MAX_NEW_TOKENS = 8

print('INPUT_CSV  :', INPUT_CSV)
print('PROMPT_JSON:', PROMPT_JSON)
print('OUTPUT_CSV :', OUTPUT_CSV)


In [ ]:
import json
import pandas as pd

def parse_yes_no(response: str):
    r = (response or '').strip().lower()
    if r.startswith('yes'):
        return 1
    if r.startswith('no'):
        return 0
    return None

def extract_prompts(prompt_json_obj):
    prompts = []
    if isinstance(prompt_json_obj, list):
        source = prompt_json_obj
    elif isinstance(prompt_json_obj, dict):
        source = []
        for _, v in prompt_json_obj.items():
            if isinstance(v, list):
                source.extend(v)
    else:
        raise ValueError('Unsupported prompt_questions.json format')

    for i, p in enumerate(source):
        if isinstance(p, dict):
            pid = p.get('id', f'prompt_{i:03d}')
            q = p.get('question', p.get('prompt', str(p)))
        else:
            pid = f'prompt_{i:03d}'
            q = str(p)
        prompts.append((pid, q))
    return prompts

def build_input(text, question):
    return (
        'You are a readability evaluator.\n'
        'Answer with one word only: yes or no.\n\n'
        f'Text:\n{text}\n\n'
        f'Question: {question}\n'
        'Answer:'
    )

assert os.path.exists(INPUT_CSV), f'Input CSV not found: {INPUT_CSV}'
assert os.path.exists(PROMPT_JSON), f'Prompt JSON not found: {PROMPT_JSON}'
df = pd.read_csv(INPUT_CSV)
assert TEXT_COL in df.columns, f'Missing text column: {TEXT_COL}'
with open(PROMPT_JSON, 'r', encoding='utf-8') as f:
    prompt_obj = json.load(f)
prompts = extract_prompts(prompt_obj)
print('Rows   :', len(df))
print('Prompts:', len(prompts))


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
)
model.eval()
print('Model loaded.')


In [ ]:
from tqdm import tqdm

@torch.no_grad()
def batched_yes_no(texts, max_new_tokens=8):
    enc = tokenizer(
        texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(model.device)

    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    parsed = [parse_yes_no(x) for x in decoded]
    return parsed, decoded

if os.path.exists(PROGRESS_CSV):
    done_df = pd.read_csv(PROGRESS_CSV)
    start_idx = len(done_df)
    results = done_df.to_dict(orient='records')
    print(f'Resuming from row {start_idx}')
else:
    start_idx = 0
    results = []
    print('Starting fresh')

for i in tqdm(range(start_idx, len(df)), desc=f'{MODEL_TAG} OOD prompt inference'):
    row = df.iloc[i].to_dict()
    text = str(row.get(TEXT_COL, ''))

    feature_values = {}
    for j in range(0, len(prompts), BATCH_SIZE):
        batch_prompts = prompts[j:j + BATCH_SIZE]
        model_inputs = [build_input(text, q) for _, q in batch_prompts]
        parsed, _ = batched_yes_no(model_inputs, max_new_tokens=MAX_NEW_TOKENS)

        for (pid, q), val in zip(batch_prompts, parsed):
            if val is None:
                retry_input = (
                    'Answer ONLY yes or no.\n'
                    f'Text:\n{text}\n\nQuestion: {q}\nAnswer:'
                )
                retry_parsed, _ = batched_yes_no([retry_input], max_new_tokens=MAX_NEW_TOKENS)
                val = retry_parsed[0]
                if val is None:
                    val = 0
            feature_values[pid] = int(val)

    out_row = dict(row)
    out_row.update(feature_values)
    results.append(out_row)

    if (i + 1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(PROGRESS_CSV, index=False)

pd.DataFrame(results).to_csv(PROGRESS_CSV, index=False)
pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
print('Done.')
print('Progress:', PROGRESS_CSV)
print('Final   :', OUTPUT_CSV)


In [ ]:
out = pd.read_csv(OUTPUT_CSV)
prompt_cols = [c for c in out.columns if c.startswith('prompt_')]
print('Shape:', out.shape)
print('Prompt columns found:', len(prompt_cols))
out.head(2)
